In [8]:
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

sys.path.append('C:/dev/project/SKN27-2nd-4TEAM')

from src.seed import set_seed                                              # ← 추가
from src.pipeline.missing_value import CJ_MissingValue, JH_train_stats, JH_MissingValue
from src.pipeline.outlier_control import OutlierControl
from src.pipeline.features import FeatureCreate

# 시드 고정
set_seed(42)  # ← 추가

In [9]:
# ── 1. 데이터 로드 ──────────────────────────────
df = pd.read_excel('C:/dev/project/SKN27-2nd-4TEAM/data/dataset.xlsx', sheet_name='E Comm')
df = df.drop(columns=['CustomerID'])  # ID 컬럼 제거

# ── 2. Train/Test 분리 ──────────────────────────
X = df.drop(columns=['Churn'])
y = df['Churn']

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_tr: {X_tr.shape}, X_te: {X_te.shape}")
print(f"Train churn 비율: {y_tr.mean():.3f}")
print(f"Test  churn 비율: {y_te.mean():.3f}")

X_tr: (4504, 18), X_te: (1126, 18)
Train churn 비율: 0.168
Test  churn 비율: 0.169


In [16]:
import numpy as np

def outlier_control(df, cols):

    df = df.copy()

    for col in cols:

        # 🔥 1. 결측치 먼저 처리
        df[col] = df[col].fillna(df[col].median())

        if col == 'CashbackAmount':
            df[f'{col}_clip'] = df[col].clip(lower=100, upper=240)

        elif col == 'DaySinceLastOrder':
            df[f'{col}_clip'] = df[col].clip(upper=18)

        else:
            # 🔥 2. 음수 방지 후 log
            safe_col = df[col].clip(lower=0)
            df[f'{col}_log'] = np.log1p(safe_col)

    # 🔥 3. inf 처리
    df = df.replace([np.inf, -np.inf], np.nan)

    # 🔥 4. 남은 결측 처리
    df = df.fillna(df.median(numeric_only=True))

    # 🔥 5. 원본 삭제
    df.drop(cols, axis=1, inplace=True)

    return df

In [13]:
# 🔥 팀원 코드 import
from src.pipeline.missing_value import JH_train_stats, JH_MissingValue
from src.pipeline.outlier_control import OutlierControl
from src.pipeline.features import FeatureCreate

# ---------------------------
# 3) 결측치 처리 (train 기준)
# ---------------------------
train_df = pd.concat([X_tr, y_tr], axis=1)
test_df  = pd.concat([X_te, y_te], axis=1)

stats = JH_train_stats(train_df)

train_df = JH_MissingValue(train_df, **stats)
test_df  = JH_MissingValue(test_df, **stats)

# ---------------------------
# 4) 이상치 처리
# ---------------------------
cols = ['Tenure','WarehouseToHome','DaySinceLastOrder','CashbackAmount']

train_df = OutlierControl(train_df, cols)
test_df  = OutlierControl(test_df, cols)

# ---------------------------
# 5) Feature Engineering
# ---------------------------
train_df = FeatureCreate(train_df)
test_df  = FeatureCreate(test_df)

# ---------------------------
# 6) 다시 X, y 분리
# ---------------------------
X_tr = train_df.drop(columns=['Churn'])
y_tr = train_df['Churn']

X_te = test_df.drop(columns=['Churn'])
y_te = test_df['Churn']

# 컬럼 맞추기 (🔥 중요)
X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)

# ---------------------------
# 7) 로지스틱 회귀
# ---------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

model = LogisticRegression(max_iter=1000)

model.fit(X_tr, y_tr)

# ---------------------------
# 8) 평가
# ---------------------------
y_pred = model.predict(X_te)
y_proba = model.predict_proba(X_te)[:, 1]

acc = accuracy_score(y_te, y_pred)
auc = roc_auc_score(y_te, y_proba)

print(f"Accuracy: {acc:.4f}")
print(f"ROC-AUC: {auc:.4f}")

KeyError: 'HourSpendOnApp'

In [15]:
train_df.isnull().sum().sort_values(ascending=False)

Dormancy_Shock                         672
Recency_Tenure_Ratio                   460
MonthlyOrderFreq                       425
Tenure_log                             213
Satisfaction_Per_Order                 212
CashbackPerOrder                       212
Promo_Sensitivity                      212
OrderAmountHikeFromlastYear            208
NumberOfDeviceRegistered                 0
CityTier                                 0
IssueIndex                               0
NumberOfAddress                          0
Silent_Killer                            0
Stagnant_Loyal                           0
ManyAddressesFlag                        0
PreferredLoginDevice_Mobile Phone        0
PreferredLoginDevice_Phone               0
PreferedOrderCat_Grocery                 0
PreferedOrderCat_Laptop & Accessory      0
PreferedOrderCat_Mobile                  0
PreferedOrderCat_Mobile Phone            0
PreferedOrderCat_Others                  0
MaritalStatus_Married                    0
MaritalStat